# Поиск нетипичного поведения заявок на закупку
Воспроизводимый анализ журнала `students.csv`. Запуск: **Runtime → Run all**; при запросе загрузите `students.csv`.
Результат: `anomalies.csv` (+ печать проверок и сводок для отчёта). Все пороги выведены из данных и пояснены в комментариях.

In [ ]:
import os, csv
import numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 120)
SEED = 42
DATA_PATH = 'students.csv'
OUT_PATH = 'anomalies.csv'
CUT = pd.Timestamp('2026-07-01 00:00', tz='+03:00')   # срез наблюдения из описания данных
if not os.path.exists(DATA_PATH):
    from google.colab import files
    files.upload()                                     # выберите students.csv

## 1. Загрузка и типы
UUID читаем как строки, кодировка `utf-8-sig` (BOM). Время приводим к tz-aware, порядок событий — строго по `EventIndex`.

In [ ]:
d = pd.read_csv(DATA_PATH, encoding='utf-8-sig', dtype=str)
d['EventIndex'] = d.EventIndex.astype(int); d['Amount'] = d.Amount.astype(float)
d['ts'] = pd.to_datetime(d.Timestamp, utc=True).dt.tz_convert('+03:00')
d = d.sort_values(['ProcessInstanceId', 'EventIndex']).reset_index(drop=True)
print('events', len(d), 'processes', d.ProcessInstanceId.nunique())
print(d.EventType.value_counts().to_dict())
print(d.drop_duplicates('ProcessInstanceId').RequestType.value_counts().to_dict())

## 2. Проверки качества данных
Каждая проверка либо печатает число, либо падает через `assert`. Расхождение со срезом фиксируем флагом, а не удаляем события.

In [ ]:
g = d.groupby('ProcessInstanceId')
mx = g.EventIndex.agg(['min', 'max', 'count'])
assert d.EventId.is_unique
assert not d.duplicated(['ProcessInstanceId', 'EventIndex']).sum()
assert ((mx['min'] == 1) & (mx['max'] == mx['count'])).all()                 # индексы 1..n без дыр
assert not (g.ts.diff() < pd.Timedelta(0)).any()                             # время не убывает по индексу
assert (g.RequestType.nunique() == 1).all() and (g.Amount.nunique() == 1).all()
assert (g.EventType.first() == 'process_started').all() and (g.EventType.nth(1).values == 'automatic_completed').all()
pc = d[d.EventType == 'process_completed'].groupby('ProcessInstanceId').size(); assert (pc == 1).all()
tc = d[d.EventType == 'task_created'].groupby('TaskId').size(); tk = d[d.EventType == 'task_completed'].groupby('TaskId').size()
assert (tc == 1).all() and (tk == 1).all() and not (set(tk.index) - set(tc.index))
print('created without close (open tasks):', len(set(tc.index) - set(tk.index)))
res = d[d.EventType == 'process_completed'].set_index('ProcessInstanceId').Result
rej = d[(d.EventType == 'task_completed') & (d.Result == 'reject')].ProcessInstanceId.unique()
assert (res.reindex(rej) == 'rejected').all() and (res == 'rejected').sum() == len(rej)
d['AfterCutoff'] = d.ts >= CUT
a = d[d.AfterCutoff]
print(f'DEFECT: events at/after cutoff: {len(a)} in {a.ProcessInstanceId.nunique()} processes, {a.ts.min()} .. {a.ts.max()}')
print('open processes (no process_completed):', d.ProcessInstanceId.nunique() - len(pc),
      '| overlap with after-cutoff processes:', len(set(a.ProcessInstanceId) - set(pc.index)))

## 3. Рабочая длительность и таблица заданий
Рабочий календарь: Пн–Пт, 09:00–18:00, без праздников и обеда. Для незакрытых заданий конец = срез (нижний предел).

In [ ]:
def work_minutes(a, b):
    if b <= a: return 0.0
    tot, day = 0.0, a.normalize()
    while day <= b.normalize():
        if day.weekday() < 5:
            s = max(a, day + pd.Timedelta(hours=9)); e = min(b, day + pd.Timedelta(hours=18))
            if e > s: tot += (e - s).total_seconds() / 60
        day += pd.Timedelta(days=1)
    return tot
T_ = lambda s: pd.Timestamp(s, tz='+03:00')
assert work_minutes(T_('2026-01-12 09:00'), T_('2026-01-12 18:00')) == 540
assert work_minutes(T_('2026-01-16 17:00'), T_('2026-01-19 10:00')) == 120     # пятница -> понедельник
assert work_minutes(T_('2026-01-17 10:00'), T_('2026-01-18 15:00')) == 0       # выходные
assert work_minutes(T_('2026-01-12 08:00'), T_('2026-01-12 08:30')) == 0       # до начала дня

cr = d[d.EventType == 'task_created'].set_index('TaskId'); cl = d[d.EventType == 'task_completed'].set_index('TaskId')
t = cr[['ProcessInstanceId', 'EventIndex', 'ActivityName', 'ts', 'RequestType', 'Amount']].rename(columns={'ts': 'created', 'EventIndex': 'ci'})
t['closed'] = cl.ts.reindex(t.index); t['ui'] = cl.UserId.reindex(t.index)
t['role'] = cl.Role.reindex(t.index); t['result'] = cl.Result.reindex(t.index)
t['open'] = t.closed.isna(); t['end'] = t.closed.fillna(CUT)
t['cal_h'] = (t.end - t.created).dt.total_seconds() / 3600
t['work_h'] = [work_minutes(x, y) / 60 for x, y in zip(t.created, t.end)]
t = t.reset_index().sort_values(['ProcessInstanceId', 'ci']).reset_index(drop=True)
assert (t.cal_h >= 0).all()
print('tasks', len(t), 'open', int(t.open.sum()))
outside = ((t.created.dt.weekday >= 5) | (t.created.dt.hour < 9) | (t.created.dt.hour >= 18)).sum()
print('tasks created outside working calendar:', outside)

## 4. Маршруты и обычное поведение (по типам заявок)
Код маршрута: CHK проверка, MGR руководитель, BUD бюджет, LEG юрист, EXE исполнение, FIX доработка, AUD внеплановая;
`+` accept, `<` return, `x` reject, `?` открыто.

In [ ]:
CODE = {'Проверка заявки': 'CHK', 'Согласование руководителем': 'MGR', 'Согласование бюджета': 'BUD', 'Проверка договора': 'LEG',
        'Исполнение заявки': 'EXE', 'Доработка заявки': 'FIX', 'Внеплановая проверка': 'AUD'}
SYM = {'accept': '+', 'return': '<', 'reject': 'x'}
t['c'] = [CODE[a] + ('?' if o else SYM[r]) for a, o, r in zip(t.ActivityName, t.open, t.result)]
fin = d[d.EventType == 'process_completed'].set_index('ProcessInstanceId').Result
P = d.drop_duplicates('ProcessInstanceId').set_index('ProcessInstanceId')[['RequestType', 'Amount']]
P = P.join(t.groupby('ProcessInstanceId').c.apply(' '.join).rename('route')).join(fin.rename('final'))
P['final'] = P.final.fillna('OPEN'); P['toks'] = P.route.str.split(); P['n_ret'] = P.route.str.count('<')
for rtp in ['standard', 'urgent', 'contract']:
    s = P[P.RequestType == rtp]; vc = s.groupby(['route', 'final']).size().sort_values(ascending=False)
    print(f'\n=== {rtp}: {len(s)} процессов, вариантов маршрута {len(vc)}'); print(vc.head(8).to_string())
print('\nвозвратов на процесс:', P.n_ret.value_counts().sort_index().to_dict())
print(pd.crosstab(t.ActivityName, t.role.fillna('OPEN')).to_string())          # этап x роль
closed = t[~t.open]
stats = closed.groupby(['RequestType', 'ActivityName']).work_h.agg(med='median', p98=lambda s: s.quantile(.98), p99=lambda s: s.quantile(.99), mx='max', n='size')
print(stats.round(2).to_string())

## 5. Правила аномалий
Нарушение = противоречие обязательной схеме (порядок, этапы, роль). Статистический хвост включаем только при резком отрыве от распределения.

In [ ]:
has = lambda x, a: any(k.startswith(a) for k in x)
pos = lambda x, a: next(i for i, k in enumerate(x) if k.startswith(a))
F = pd.DataFrame(index=P.index)
F['skip_mgr'] = P.toks.apply(lambda x: has(x, 'BUD') and not has(x, 'MGR'))                       # бюджет без руководителя
F['exe_before_bud'] = P.toks.apply(lambda x: has(x, 'EXE') and has(x, 'BUD') and pos(x, 'EXE') < pos(x, 'BUD'))
F['dbl_chk'] = P.toks.apply(lambda x: any(x[i].startswith('CHK') and x[i + 1].startswith('CHK') for i in range(len(x) - 1)))
F['audit'] = P.toks.apply(lambda x: has(x, 'AUD'))
F['many_ret'] = P.n_ret >= 4        # в журнале 0 и 1 возврат (992 заявки), затем сразу 5-8: разрыв 2..4
F['req_as_mgr'] = F.index.isin(t[(t.ActivityName == 'Согласование руководителем') & (t.role == 'requester')].ProcessInstanceId)
F['instant'] = F.index.isin(closed[closed.cal_h * 3600 < 60].ProcessInstanceId)                   # задание закрыто быстрее минуты
chk = closed[closed.ActivityName == 'Проверка заявки']
print('проверка заявки, значения work_h > 2.5 ч:', sorted(chk[chk.work_h > 2.5].work_h.round(2).tolist()))   # между 2.5 и 8 ч пусто
F['stuck_chk'] = F.index.isin(chk[chk.work_h >= 30].ProcessInstanceId)                            # 60 рабочих часов
F['slow_chk'] = F.index.isin(chk[(chk.work_h > 2.5) & (chk.work_h < 30)].ProcessInstanceId)       # 8 рабочих часов
# открытое задание: нижний предел ожидания больше 3 x p99 закрытых заданий того же типа и этапа
p99 = closed.groupby(['RequestType', 'ActivityName']).work_h.quantile(.99)
op = t[t.open].copy(); op['p99'] = [p99[(r, a)] for r, a in zip(op.RequestType, op.ActivityName)]
print(op[['ProcessInstanceId', 'ActivityName', 'work_h', 'p99']].round(2).to_string())
F['stuck_open'] = F.index.isin(op[op.work_h > 3 * op.p99].ProcessInstanceId)
print(F.sum().to_string()); print('flagged:', int(F.any(axis=1).sum()))
print(F[F.sum(axis=1) > 1].apply(lambda r: ','.join(r.index[r]), axis=1).value_counts().to_string())

## 6. Перекрёстная проверка: Isolation Forest
Не основной метод, а независимый контроль: смотрим, что он видит и что пропускает правилами.

In [ ]:
g = t.groupby('ProcessInstanceId')
X = pd.DataFrame({'ntask': g.size(), 'nret': P.n_ret, 'work_tot': np.log1p(g.work_h.sum()), 'work_max': np.log1p(g.work_h.max()),
                  'cal_tot': np.log1p(g.cal_h.sum()), 'minlog': np.log10(g.cal_h.min() * 3600 + 1), 'nuser': g.ui.nunique(),
                  'nauditor': g.role.apply(lambda s: (s == 'auditor').sum()), 'nreq': g.role.apply(lambda s: (s == 'requester').sum()),
                  'amount': np.log10(P.Amount)})
vf = P.groupby(['RequestType', 'route']).size(); tot = P.groupby('RequestType').size()
X['rare_route'] = [-np.log(vf[(a, b)] / tot[a]) for a, b in zip(P.RequestType, P.route)]
X = pd.concat([X, pd.get_dummies(P.RequestType).astype(float)], axis=1)
for col in ['work_tot', 'work_max', 'cal_tot']: X[col] = X[col] - X.groupby(P.RequestType)[col].transform('median')
score = pd.Series(-IsolationForest(n_estimators=500, random_state=SEED).fit(X).score_samples(X), index=X.index)
rank = score.rank(ascending=False); rule = F.any(axis=1)
for K in (60, 100):
    top = rank <= K; print(f'top-{K}: в правилах {(top & rule).sum()}, только IF {(top & ~rule).sum()}, правила вне топа {(~top & rule).sum()}')
print('медианный ранг флагов по классам:', {c: int(rank[F[c]].median()) for c in F.columns if F[c].any()})

## 7. Объяснения и `anomalies.csv`

In [ ]:
fmt = lambda x: f'{x:.2f}'.replace('.', ',')
tsf = lambda x: x.strftime('%d.%m.%Y %H:%M')
SH = {'Проверка заявки': 'проверка заявки', 'Согласование руководителем': 'согласование руководителем', 'Согласование бюджета': 'согласование бюджета',
      'Проверка договора': 'проверка договора', 'Исполнение заявки': 'исполнение заявки', 'Доработка заявки': 'доработка', 'Внеплановая проверка': 'внеплановая проверка'}
st = closed.groupby(['RequestType', 'ActivityName']).work_h.agg(med='median', p99=lambda s: s.quantile(.99))
TT = {k: v.sort_values('ci') for k, v in t.groupby('ProcessInstanceId')}
rt = lambda g_: ' → '.join(SH[a] for a in g_.ActivityName)
n0, n1 = int((P.n_ret == 0).sum()), int((P.n_ret == 1).sum()); n_ok = n0 + n1
mgr_closed = closed[closed.ActivityName == 'Согласование руководителем']
n_mgr, n_mgr_ok = len(mgr_closed), int((mgr_closed.role == 'manager').sum())
n_aud = int(F.audit.sum()); n_all = len(P)
rows = []
add = lambda pid, typ, txt: rows.append((pid, typ, txt))
for pid in P.index[F.skip_mgr]:
    g_ = TT[pid]; rtp = P.RequestType[pid]
    reach = P[(P.RequestType == rtp) & P.route.str.contains('BUD')]; withm = int(reach.route.str.contains('MGR').sum())
    add(pid, 'Skipped manager approval', f'Заявка типа {rtp}: маршрут {rt(g_)}. После проверки заявки сразу создано согласование бюджета, этапа «согласование руководителем» нет вовсе, хотя это обязательный этап. Среди заявок типа {rtp}, дошедших до бюджета, руководитель отсутствует лишь в {len(reach) - withm} из {len(reach)}, в остальных {withm} он есть.')
for pid in P.index[F.exe_before_bud]:
    g_ = TT[pid]; rtp = P.RequestType[pid]; e = g_[g_.ActivityName == 'Исполнение заявки'].iloc[0]; b = g_[g_.ActivityName == 'Согласование бюджета'].iloc[0]
    extra = ''
    if rtp == 'contract':
        l = g_[g_.ActivityName == 'Проверка договора'].iloc[0]; extra = f' и юридической проверки (создана {tsf(l.created)})'
    add(pid, 'Execution before budget approval', f'Заявка типа {rtp}: маршрут {rt(g_)}. Исполнение закупщиком (создано {tsf(e.created)}, закрыто {tsf(e.closed)}) выполнено раньше согласования бюджета (создано {tsf(b.created)}){extra}. По схеме бюджет (и юрист для договорной заявки) должен предшествовать исполнению; в остальных заявках исполнение всегда последнее.')
for pid in P.index[F.dbl_chk]:
    g_ = TT[pid]; k = g_[g_.ActivityName == 'Проверка заявки']; a_, b_ = k.iloc[0], k.iloc[1]
    assert a_.result == 'accept' and b_.result == 'accept'
    add(pid, 'Duplicate specialist check', f'Заявка типа {P.RequestType[pid]}: проверка заявки выполнена два раза подряд (закрыта {tsf(a_.closed)} и {tsf(b_.closed)}), оба раза accept, возврата на доработку между ними нет. Повторная проверка допустима только после возврата, здесь её причина в данных не видна. У остальных заявок без возврата проверка выполняется ровно один раз.')
for pid in P.index[F.audit]:
    g_ = TT[pid]; a_ = g_[g_.ActivityName == 'Внеплановая проверка']
    add(pid, 'Unplanned audit checks', f'Заявка типа {P.RequestType[pid]}: две внеплановые проверки сотрудником контроля (auditor), до и после согласования бюджета; обе закрыты accept (рабочая длительность {fmt(a_.work_h.iloc[0])} и {fmt(a_.work_h.iloc[1])} ч, исполнителей: {a_.ui.nunique()}). Маршрут {rt(g_)}. Оправдание проверок по данным установить нельзя: возврата или отказа до них нет, результат не изменил маршрут. Всего в журнале {n_aud} таких заявок из {n_all}. Требует ручной проверки бизнес-причины.')
for pid in P.index[F.req_as_mgr]:
    g_ = TT[pid]; x = g_[(g_.ActivityName == 'Согласование руководителем') & (g_.role == 'requester')].iloc[0]
    add(pid, 'Manager task performed by requester', f'Заявка типа {P.RequestType[pid]}: согласование руководителем закрыто пользователем с ролью requester (результат {x.result}, закрыто {tsf(x.closed)}), а не manager. Из {n_mgr} закрытых заданий руководителя в журнале в {n_mgr_ok} роль manager, и только в этих {n_mgr - n_mgr_ok} заданиях другая. Сотрудник на этом этапе не имеет роли руководителя.')
for pid in P.index[F.many_ret]:
    g_ = TT[pid]; n = int(P.n_ret[pid])
    add(pid, 'Excessive rework loops', f'Заявка типа {P.RequestType[pid]} возвращалась руководителем на доработку {n} раз (всего {len(g_)} заданий, суммарная рабочая длительность {fmt(g_.work_h.sum())} ч) и затем была исполнена. У остальных {n_ok} заявок возвратов либо нет ({n0} заявок), либо один ({n1} заявок), промежуточных значений от 2 до 4 нет.')
for pid in P.index[F.instant]:
    g_ = TT[pid]; fast = g_[g_.cal_h * 3600 < 60]
    sec = ', '.join(f'{SH[a]} {round(h * 3600)} с' for a, h in zip(fast.ActivityName, fast.cal_h))
    add(pid, 'Instant task chain', f'Заявка типа {P.RequestType[pid]}: {len(fast)} из {len(g_)} заданий закрыты за секунды ({sec}). Для сравнения, обычная рабочая длительность того же этапа у заявок этого типа от минут до часов (например, медиана согласования руководителем {fmt(st.loc[(P.RequestType[pid], "Согласование руководителем"), "med"])} ч). Такой темп похож на формальное, пакетное закрытие без реальной проверки.')
for pid in P.index[F.stuck_chk]:
    g_ = TT[pid]; k = g_[g_.ActivityName == 'Проверка заявки'].iloc[0]; rtp = P.RequestType[pid]
    add(pid, 'Stalled specialist check', f'Заявка типа {rtp}: проверка заявки длилась {fmt(k.work_h)} рабочих ч ({fmt(k.cal_h)} календарных) при медиане {fmt(st.loc[(rtp, "Проверка заявки"), "med"])} ч и 99-м перцентиле {fmt(st.loc[(rtp, "Проверка заявки"), "p99"])} ч по заявкам этого типа. Значение ровно 60 рабочих часов (около 6,7 рабочих дня при 9-часовом дне), как и у остальных семи таких заявок, похоже на длительное простаивание задания.')
for pid in P.index[F.slow_chk]:
    g_ = TT[pid]; k = g_[g_.ActivityName == 'Проверка заявки'].iloc[0]; rtp = P.RequestType[pid]
    p98 = closed[(closed.RequestType == rtp) & (closed.ActivityName == 'Проверка заявки')].work_h.quantile(.98)
    add(pid, 'Slow specialist check', f'Заявка типа {rtp}: проверка заявки заняла ровно {fmt(k.work_h)} рабочих ч ({fmt(k.cal_h)} календарных) при медиане {fmt(st.loc[(rtp, "Проверка заявки"), "med"])} ч; 98-й перцентиль среди {rtp} равен {fmt(p98)} ч, между 2,5 и 8 ч заданий проверки нет. Значение примерно в {round(k.work_h / st.loc[(rtp, "Проверка заявки"), "med"])} раз выше медианы и одинаково у {int(F.slow_chk.sum())} заявок.')
for pid in P.index[F.stuck_open]:
    g_ = TT[pid]; o = g_[g_.open].iloc[0]; rtp = P.RequestType[pid]
    add(pid, 'Stalled open manager approval', f'Заявка типа {rtp} не завершена: задание согласования руководителем создано {tsf(o.created)} и до среза (01.07.2026) не закрыто, то есть длится не менее {fmt(o.work_h)} рабочих ч ({fmt(o.cal_h)} календарных). Обычно этот этап занимает медиану {fmt(st.loc[(rtp, "Согласование руководителем"), "med"])} ч, 99-й перцентиль {fmt(st.loc[(rtp, "Согласование руководителем"), "p99"])} ч. Открытость сама по себе не аномалия, но ожидание превышает норму более чем в 100 раз.')
out = pd.DataFrame(rows, columns=['ProcessInstanceId', 'AnomalyType', 'Explanation']).drop_duplicates()
out.to_csv(OUT_PATH, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
print(len(out), 'строк,', out.ProcessInstanceId.nunique(), 'процессов'); print(out.AnomalyType.value_counts().to_string())

## 8. Валидация файла

In [ ]:
chk_df = pd.read_csv(OUT_PATH, encoding='utf-8', dtype=str)
assert list(chk_df.columns) == ['ProcessInstanceId', 'AnomalyType', 'Explanation']
assert chk_df.ProcessInstanceId.isin(d.ProcessInstanceId).all()
assert not chk_df.duplicated().any() and not chk_df.isna().any().any()
assert len(list(csv.reader(open(OUT_PATH, encoding='utf-8')))) - 1 == len(chk_df)
print('OK:', chk_df.shape)
# слабые места для ручной проверки: хвосты без разрыва (не включены в CSV)
tail = closed[(closed.RequestType == 'standard') & (closed.ActivityName == 'Исполнение заявки')].nlargest(3, 'work_h')[['ProcessInstanceId', 'work_h']]
print(tail.round(2).to_string())